# LIBERO 학습 — **노드 B** (GPU 4장)

우리 클러스터는 노드 3대(**A=2 · B=4 · C=4** GPU, 합 10)로 LIBERO 를 나눠 돌린다.
이 노트북은 **노드 B(4 GPU)** 몫만 학습한다. 세 노드에서 각자 `train_node{A,B,C}` 를 열어
동시에 돌리면 6모델×4seed=24잡이 GPU 비율(2:4:4)대로 분배된다.

- task = `libero_10` (LIBERO-LONG, 520스텝) · 150k step · lr 고정.
- 잡은 **(모델,seed)** 단위로 독립 → 어느 노드가 돌리든 결과는 같은 경로에 쌓여 리포트가 자동 pooled.
- ⚠️ **학습**은 데이터셋만 있으면 된다(시뮬 불필요). **eval** 은 LIBERO 시뮬 필요 → `eval_node{node}` 에서.
- resume 자동: 끊겨도 다시 실행하면 마지막 체크포인트부터 이어간다.


In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

In [ ]:
NODE = 'B'
# 이 노드가 맡을 (모델, seed, task) 잡 — GPU 비율(2:4:4)로 자동 분배된 몫
jobs = cf.node_jobs(NODE)
gpus = cf.available_gpus()[:cf.NODE_GPUS[NODE]]   # 이 노드에 보이는 GPU 를 0번부터
print(f'노드 {NODE} | GPU {gpus} | {len(jobs)}잡')
for t, s, tk in jobs:
    print(f'   {t:10} seed{s}  ({tk})')

## 학습 실행 — resume/skip/청크 자동
GPU 수만큼 동시에 띄우고, 청크가 끝날 때까지 대기 후 다음 청크. 이미 150k 끝난 잡은 skip.


In [ ]:
cf.run_training_jobs(jobs, gpus=gpus)   # prefetch(데이터셋 1회 선다운) 포함

## 진행 상태


In [ ]:
for t, s, tk in jobs:
    out = cf.v23.train_dir(t, s, tk)
    step = cf.v23.last_ckpt_step(out)
    done = '완료' if (step or 0) >= cf.CKPT_STEP else (f'{step:,}' if step else '아직')
    print(f'   {t:10} seed{s}: {done}')